In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

property_2021 = pd.read_parquet(
    "/content/drive/MyDrive/GNN/property_features_2021_checkpoint.parquet"
)

eviction_2021 = pd.read_parquet(
    "/content/drive/MyDrive/GNN/eviction_outcomes_2021.parquet"
)

print("Property dataset:", property_2021.shape)
print("Eviction outcomes:", eviction_2021.shape)

print("\nProperty columns:")
print(property_2021.columns.tolist())

print("\nEviction columns:")
print(eviction_2021.columns.tolist())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Property dataset: (1507914, 117)
Eviction outcomes: (8487, 4)

Property columns:
['acct', 'yr', 'mailto', 'mail_addr_1', 'mail_addr_2', 'mail_city', 'mail_state', 'mail_zip', 'site_addr_1', 'site_addr_2', 'site_addr_3', 'state_class', 'Neighborhood_Code', 'Neighborhood_Grp', 'Market_Area_1', 'Market_Area_1_Dscr', 'yr_impr', 'bld_ar', 'land_ar', 'acreage', 'land_val', 'bld_val', 'tot_appr_val', 'tot_mkt_val', 'new_own_dt', 'explicit_rental', 'explicit_rental_building', 'explicit_rental_state', 'explicit_rental_pgi', 'in_building_res', 'in_building_other', 'exempt_cat', 'has_res_homestead', 'has_partial_homestead', 'has_apportioned_residential', 'has_low_income_housing', 'any_homestead_evidence', 'has_corporate_owner', 'fixture_building_count', 'max_stories', 'mean_stories', 'bedrooms', 'full_baths', 'half_baths', 'rec_rooms', 'total_rooms', 'efficiency_units',

In [5]:
import pandas as pd

property_2021 = pd.read_parquet(
    "/content/drive/MyDrive/GNN/property_features_2021_checkpoint.parquet"
)

eviction_2021 = pd.read_parquet(
    "/content/drive/MyDrive/GNN/eviction_outcomes_2021.parquet"
)

print("Property dataset:", property_2021.shape)
print("Eviction outcomes:", eviction_2021.shape)

Property dataset: (1507914, 117)
Eviction outcomes: (8487, 4)


In [6]:
property_2021["acct"] = (
    property_2021["acct"]
    .astype("string")
    .str.strip()
    .str.zfill(13)
)

eviction_2021["acct"] = (
    eviction_2021["acct"]
    .astype("string")
    .str.strip()
    .str.zfill(13)
)

In [7]:
eviction_account_check = (
    eviction_2021[["acct"]]
    .merge(
        property_2021[["acct"]].drop_duplicates(),
        on="acct",
        how="left",
        indicator=True
    )
)

print(
    "Eviction properties:",
    eviction_2021["acct"].nunique()
)

print(
    "Found in property dataset:",
    (eviction_account_check["_merge"] == "both").sum()
)

print(
    "Not found in property dataset:",
    (eviction_account_check["_merge"] == "left_only").sum()
)

print(
    "Duplicate property accounts in checkpoint:",
    property_2021["acct"].duplicated().sum()
)

Eviction properties: 8487
Found in property dataset: 8487
Not found in property dataset: 0
Duplicate property accounts in checkpoint: 0


In [8]:
analysis_2021 = property_2021.merge(
    eviction_2021[
        [
            "acct",
            "filing_count_2021",
            "any_filing_2021"
        ]
    ],
    on="acct",
    how="left"
)

analysis_2021["filing_count_2021"] = (
    analysis_2021["filing_count_2021"]
    .fillna(0)
    .astype("int32")
)

analysis_2021["any_filing_2021"] = (
    analysis_2021["any_filing_2021"]
    .fillna(0)
    .astype("int8")
)

print("Rows:", len(analysis_2021))
print("Unique accounts:", analysis_2021["acct"].nunique())

print(
    "Properties with >=1 matched filing:",
    analysis_2021["any_filing_2021"].sum()
)

print(
    "Total matched filings:",
    analysis_2021["filing_count_2021"].sum()
)

Rows: 1507914
Unique accounts: 1507914
Properties with >=1 matched filing: 8487
Total matched filings: 26734


In [9]:
print("All HCAD properties:")
print(
    analysis_2021["explicit_rental"]
    .value_counts(dropna=False)
)

print("\nProperties with >=1 matched eviction filing:")
print(
    analysis_2021.loc[
        analysis_2021["any_filing_2021"] == 1,
        "explicit_rental"
    ]
    .value_counts(dropna=False)
)

print("\nMatched filings by explicit_rental status:")
print(
    analysis_2021
    .groupby("explicit_rental", dropna=False)["filing_count_2021"]
    .sum()
)

All HCAD properties:
explicit_rental
0    1491617
1      16297
Name: count, dtype: int64

Properties with >=1 matched eviction filing:
explicit_rental
0    6268
1    2219
Name: count, dtype: int64

Matched filings by explicit_rental status:
explicit_rental
0     7981
1    18753
Name: filing_count_2021, dtype: int32


In [10]:
eviction_properties = analysis_2021[
    analysis_2021["any_filing_2021"] == 1
]

print(
    "Percent of eviction properties explicitly classified as rental:",
    round(
        eviction_properties["explicit_rental"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Percent of matched filings occurring at explicit-rental properties:",
    round(
        eviction_properties.loc[
            eviction_properties["explicit_rental"] == 1,
            "filing_count_2021"
        ].sum()
        /
        eviction_properties["filing_count_2021"].sum()
        * 100,
        2
    ),
    "%"
)

Percent of eviction properties explicitly classified as rental: 26.15 %
Percent of matched filings occurring at explicit-rental properties: 70.15 %


In [11]:
filing_properties_2021 = (
    analysis_2021[
        analysis_2021["filing_count_2021"] > 0
    ]
    .copy()
)

print(
    filing_properties_2021["filing_count_2021"]
    .describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    )
)

print("\nProperties by filing count:")
print(
    filing_properties_2021["filing_count_2021"]
    .value_counts()
    .sort_index()
    .head(20)
)

print(
    "\nMaximum filings at one property:",
    filing_properties_2021["filing_count_2021"].max()
)

count    8487.000000
mean        3.149994
std         7.280806
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
90%         6.000000
95%        14.000000
99%        37.000000
max       144.000000
Name: filing_count_2021, dtype: float64

Properties by filing count:
filing_count_2021
1     6030
2     1021
3      293
4      142
5      102
6       97
7       74
8       64
9       50
10      56
11      39
12      39
13      33
14      34
15      25
16      18
17      24
18      18
19      24
20      18
Name: count, dtype: int64

Maximum filings at one property: 144


In [12]:
print(
    "Exactly 1 filing:",
    (filing_properties_2021["filing_count_2021"] == 1).sum()
)

print(
    "2-4 filings:",
    (
        filing_properties_2021["filing_count_2021"]
        .between(2, 4)
    ).sum()
)

print(
    "5-9 filings:",
    (
        filing_properties_2021["filing_count_2021"]
        .between(5, 9)
    ).sum()
)

print(
    "10+ filings:",
    (
        filing_properties_2021["filing_count_2021"] >= 10
    ).sum()
)

print(
    "25+ filings:",
    (
        filing_properties_2021["filing_count_2021"] >= 25
    ).sum()
)

Exactly 1 filing: 6030
2-4 filings: 1456
5-9 filings: 387
10+ filings: 614
25+ filings: 218


In [13]:
filing_properties_2021["filing_group"] = pd.cut(
    filing_properties_2021["filing_count_2021"],
    bins=[0, 1, 4, 9, 24, float("inf")],
    labels=[
        "1 filing",
        "2-4 filings",
        "5-9 filings",
        "10-24 filings",
        "25+ filings"
    ]
)

filing_concentration = (
    filing_properties_2021
    .groupby("filing_group", observed=True)
    .agg(
        properties=("acct", "nunique"),
        filings=("filing_count_2021", "sum")
    )
    .reset_index()
)

filing_concentration["percent_properties"] = (
    filing_concentration["properties"]
    / filing_concentration["properties"].sum()
    * 100
)

filing_concentration["percent_filings"] = (
    filing_concentration["filings"]
    / filing_concentration["filings"].sum()
    * 100
)

display(filing_concentration)

,filing_group,properties,filings,percent_properties,percent_filings
0,1 filing,6030,6030,71.049841,22.555547
1,2-4 filings,1456,3489,17.155650,13.050797
2,5-9 filings,387,2572,4.559915,9.620708
3,10-24 filings,396,6098,4.665960,22.809905
4,25+ filings,218,8545,2.568634,31.963043


In [14]:
rental_by_filing_group = (
    filing_properties_2021
    .groupby(
        ["filing_group", "explicit_rental"],
        observed=True
    )
    .agg(
        properties=("acct", "nunique"),
        filings=("filing_count_2021", "sum")
    )
    .reset_index()
)

display(rental_by_filing_group)

,filing_group,explicit_rental,properties,filings
0,1 filing,0,5331,5331
1,1 filing,1,699,699
2,2-4 filings,0,884,1932
3,2-4 filings,1,572,1557
4,5-9 filings,0,27,183
5,5-9 filings,1,360,2389
6,10-24 filings,0,18,260
7,10-24 filings,1,378,5838
8,25+ filings,0,8,275
9,25+ filings,1,210,8270


In [15]:
rental_group_summary = (
    filing_properties_2021
    .groupby("filing_group", observed=True)
    .agg(
        properties=("acct", "nunique"),
        explicit_rental_properties=(
            "explicit_rental",
            "sum"
        ),
        filings=("filing_count_2021", "sum"),
        explicit_rental_filings=(
            "filing_count_2021",
            lambda x: x[
                filing_properties_2021.loc[
                    x.index,
                    "explicit_rental"
                ] == 1
            ].sum()
        )
    )
    .reset_index()
)

rental_group_summary["pct_properties_explicit_rental"] = (
    rental_group_summary["explicit_rental_properties"]
    / rental_group_summary["properties"]
    * 100
)

rental_group_summary["pct_filings_explicit_rental"] = (
    rental_group_summary["explicit_rental_filings"]
    / rental_group_summary["filings"]
    * 100
)

display(rental_group_summary)

,filing_group,properties,explicit_rental_properties,filings,explicit_rental_filings,pct_properties_explicit_rental,pct_filings_explicit_rental
0,1 filing,6030,699,6030,699,11.592040,11.592040
1,2-4 filings,1456,572,3489,1557,39.285714,44.625967
2,5-9 filings,387,360,2572,2389,93.023256,92.884914
3,10-24 filings,396,378,6098,5838,95.454545,95.736307
4,25+ filings,218,210,8545,8270,96.330275,96.781744


In [16]:
for col in property_2021.columns:
    if any(word in col.lower() for word in [
        "unit",
        "apt",
        "apartment",
        "living",
        "building",
        "bldg",
        "area",
        "sqft",
        "square"
    ]):
        print(col)

Market_Area_1
Market_Area_1_Dscr
explicit_rental_building
in_building_res
in_building_other
fixture_building_count
efficiency_units
one_bedroom_units
two_bedroom_units
three_bedroom_units
four_bedroom_units
structural_building_count
market_value_per_bld_sqft
max_reported_units
sum_building_units
units_from_type
estimated_units_prelim
market_value_per_unit


In [17]:
unit_cols = [
    "efficiency_units",
    "one_bedroom_units",
    "two_bedroom_units",
    "three_bedroom_units",
    "four_bedroom_units",
    "max_reported_units",
    "sum_building_units",
    "units_from_type",
    "estimated_units_prelim"
]

unit_summary = []

for col in unit_cols:
    s = pd.to_numeric(
        analysis_2021[col],
        errors="coerce"
    )

    unit_summary.append({
        "variable": col,
        "non_missing": s.notna().sum(),
        "positive": (s > 0).sum(),
        "zero": (s == 0).sum(),
        "median_positive": s[s > 0].median(),
        "p95_positive": s[s > 0].quantile(.95),
        "max": s.max()
    })

unit_summary = pd.DataFrame(unit_summary)

display(unit_summary)

,variable,non_missing,positive,zero,median_positive,p95_positive,max
0,efficiency_units,488,470,18,17.5,132.0,508.0
1,one_bedroom_units,3061,2065,996,100.0,268.8,1450.0
2,two_bedroom_units,2959,2065,894,80.0,184.0,504.0
3,three_bedroom_units,1109,940,169,20.0,84.0,355.0
4,four_bedroom_units,91,82,9,12.0,77.8,200.0
5,max_reported_units,5984,5984,0,18.5,364.0,1792.0
6,sum_building_units,16297,5984,10313,45.0,8644.4,311808.0
7,units_from_type,9995,9995,0,2.0,3.0,4.0
8,estimated_units_prelim,15978,15978,0,2.0,278.0,1792.0


In [18]:
filing_unit_summary = []

filing_props = analysis_2021[
    analysis_2021["any_filing_2021"] == 1
]

for col in unit_cols:
    s = pd.to_numeric(
        filing_props[col],
        errors="coerce"
    )

    filing_unit_summary.append({
        "variable": col,
        "non_missing": s.notna().sum(),
        "positive": (s > 0).sum(),
        "zero": (s == 0).sum(),
        "median_positive": s[s > 0].median(),
        "p95_positive": s[s > 0].quantile(.95),
        "max": s.max()
    })

filing_unit_summary = pd.DataFrame(
    filing_unit_summary
)

display(filing_unit_summary)

,variable,non_missing,positive,zero,median_positive,p95_positive,max
0,efficiency_units,246,237,9,24.0,147.20,508.0
1,one_bedroom_units,1417,1212,205,120.0,269.45,643.0
2,two_bedroom_units,1418,1219,199,90.0,184.10,434.0
3,three_bedroom_units,645,578,67,23.0,86.00,355.0
4,four_bedroom_units,57,54,3,10.5,59.80,200.0
5,max_reported_units,1871,1871,0,200.0,436.00,1326.0
6,sum_building_units,2219,1871,348,2044.0,13648.00,198900.0
7,units_from_type,303,303,0,2.0,3.00,4.0
8,estimated_units_prelim,2174,2174,0,162.0,423.00,1326.0


In [19]:
cols = [
    "acct",
    "explicit_rental",
    "filing_count_2021",
    "max_reported_units",
    "units_from_type",
    "estimated_units_prelim",
    "sum_building_units"
]

display(
    analysis_2021.loc[
        analysis_2021["any_filing_2021"] == 1,
        cols
    ].head(30)
)

,acct,explicit_rental,filing_count_2021,max_reported_units,units_from_type,estimated_units_prelim,sum_building_units
107,0010280000012,1,2,57.0,NaN,57.0,57.0
137,0010350000001,1,10,463.0,NaN,463.0,926.0
144,0010420000006,1,1,79.0,NaN,79.0,237.0
182,0010490000004,0,3,NaN,NaN,NaN,NaN
210,0010570000009,1,2,308.0,NaN,308.0,616.0
255,0010790000001,1,26,286.0,NaN,286.0,572.0
370,0011280000001,1,1,347.0,NaN,347.0,347.0
799,0020320000001,1,1,32.0,NaN,32.0,64.0
915,0020670000004,1,47,392.0,NaN,392.0,784.0
920,0020680000006,1,9,336.0,NaN,336.0,2352.0


In [20]:
s = pd.to_numeric(
    analysis_2021.loc[
        analysis_2021["any_filing_2021"] == 1,
        "estimated_units_prelim"
    ],
    errors="coerce"
)

print("Non-missing:", s.notna().sum())
print("Positive:", (s > 0).sum())

print(
    s[s > 0].describe(
        percentiles=[.25, .50, .75, .90, .95, .99]
    )
)

Non-missing: 2174
Positive: 2174
count    2174.000000
mean      174.227231
std       159.286056
min         1.000000
25%        16.000000
50%       162.000000
75%       280.000000
90%       360.000000
95%       423.000000
99%       682.160000
max      1326.000000
Name: estimated_units_prelim, dtype: float64


In [21]:
unit_compare = (
    analysis_2021.loc[
        (analysis_2021["any_filing_2021"] == 1)
        &
        analysis_2021["explicit_rental"].eq(1),
        [
            "acct",
            "max_reported_units",
            "units_from_type",
            "estimated_units_prelim",
            "sum_building_units"
        ]
    ]
    .copy()
)

for col in [
    "max_reported_units",
    "units_from_type",
    "estimated_units_prelim",
    "sum_building_units"
]:
    unit_compare[col] = pd.to_numeric(
        unit_compare[col],
        errors="coerce"
    )

display(
    unit_compare[
        [
            "max_reported_units",
            "units_from_type",
            "estimated_units_prelim",
            "sum_building_units"
        ]
    ].corr()
)

,max_reported_units,units_from_type,estimated_units_prelim,sum_building_units
max_reported_units,1.000000,NaN,1.000000,0.714889
units_from_type,NaN,1.0,1.000000,NaN
estimated_units_prelim,1.000000,1.0,1.000000,0.710289
sum_building_units,0.714889,NaN,0.710289,1.000000


In [22]:
for col in analysis_2021.columns:
    if any(word in col.lower() for word in [
        "state",
        "class",
        "pgi",
        "type",
        "res",
        "property",
        "building"
    ]):
        print(col)

mail_state
state_class
explicit_rental_building
explicit_rental_state
explicit_rental_pgi
in_building_res
in_building_other
has_res_homestead
has_apportioned_residential
fixture_building_count
foundation_type_count
hvac_type_count
exterior_wall_type_count
structural_building_count
structural_element_types
exterior_wall_type_count_missing
foundation_type_count_missing
property_age
sum_building_units
units_from_type


In [23]:
filing_nonexplicit = analysis_2021[
    (analysis_2021["any_filing_2021"] == 1)
    &
    (analysis_2021["explicit_rental"] == 0)
].copy()

print("Non-explicit properties with filings:", len(filing_nonexplicit))

for col in [
    "in_building_res",
    "in_building_other",
    "has_res_homestead",
    "has_apportioned_residential",
    "units_from_type"
]:
    print(f"\n{col}")
    print(
        filing_nonexplicit[col]
        .value_counts(dropna=False)
        .head(20)
    )

Non-explicit properties with filings: 6268

in_building_res
in_building_res
1.0    5919
0.0     191
NaN     158
Name: count, dtype: int64

in_building_other
in_building_other
0.0    5919
1.0     191
NaN     158
Name: count, dtype: int64

has_res_homestead
has_res_homestead
0    5544
1     724
Name: count, dtype: int64

has_apportioned_residential
has_apportioned_residential
0    6268
Name: count, dtype: int64

units_from_type
units_from_type
NaN    6268
Name: count, dtype: int64


In [24]:
print(
    filing_nonexplicit["state_class"]
    .value_counts(dropna=False)
    .head(30)
)

state_class
A1    5194
Z4     218
F1     209
M3     177
Z1     109
Z3      86
A2      54
O2      38
A4      28
C2      28
A3      27
X1      23
C1      19
Z5      15
C3      11
O1      10
X2       7
B1       6
B3       3
B2       3
X3       3
Name: count, dtype: Int64


In [25]:
for col in analysis_2021.columns:
    if any(word in col.lower() for word in [
        "owner",
        "mail",
        "addr",
        "address",
        "site",
        "situs",
        "homestead"
    ]):
        print(col)

mailto
mail_addr_1
mail_addr_2
mail_city
mail_state
mail_zip
site_addr_1
site_addr_2
site_addr_3
has_res_homestead
has_partial_homestead
any_homestead_evidence
has_corporate_owner


In [26]:
print("Total A1 properties:",
      (analysis_2021["state_class"] == "A1").sum())

print("A1 with no homestead:",
      (
          (analysis_2021["state_class"] == "A1")
          &
          (analysis_2021["has_res_homestead"] == 0)
      ).sum())

print("A1 with homestead:",
      (
          (analysis_2021["state_class"] == "A1")
          &
          (analysis_2021["has_res_homestead"] == 1)
      ).sum())

Total A1 properties: 1078901
A1 with no homestead: 293752
A1 with homestead: 785149


In [27]:
analysis_2021["sf_no_homestead"] = (
    analysis_2021["state_class"].eq("A1")
    &
    analysis_2021["in_building_res"].eq(1)
    &
    analysis_2021["has_res_homestead"].eq(0)
)

In [28]:
sf_candidates = (
    analysis_2021[
        (analysis_2021["state_class"] == "A1")
        &
        (analysis_2021["in_building_res"] == 1)
    ]
    .copy()
)

print("A1 residential properties:", len(sf_candidates))

print(
    "A1 with NO homestead evidence:",
    (sf_candidates["any_homestead_evidence"] == 0).sum()
)

print(
    "A1 with ANY homestead evidence:",
    (sf_candidates["any_homestead_evidence"] == 1).sum()
)

print(
    "\nCorporate-owned among A1 with no homestead:",
    (
        (sf_candidates["any_homestead_evidence"] == 0)
        &
        (sf_candidates["has_corporate_owner"] == 1)
    ).sum()
)

A1 residential properties: 1075996
A1 with NO homestead evidence: 292786
A1 with ANY homestead evidence: 783210

Corporate-owned among A1 with no homestead: 44403


In [29]:
display(
    sf_candidates.loc[
        sf_candidates["any_homestead_evidence"] == 0,
        [
            "acct",
            "mail_addr_1",
            "mail_addr_2",
            "mail_city",
            "mail_state",
            "mail_zip",
            "site_addr_1",
            "site_addr_2",
            "site_addr_3",
            "has_corporate_owner",
            "any_homestead_evidence"
        ]
    ].head(25)
)

,acct,mail_addr_1,mail_addr_2,mail_city,mail_state,mail_zip,site_addr_1,site_addr_2,site_addr_3,has_corporate_owner,any_homestead_evidence
927,0020720000014,2825 WILCREST DR STE 669,<NA>,HOUSTON,TX,77042-6043,1700 ST CHARLES ST,HOUSTON,77003,0,0
1085,0021440000008,9630 KIRKSTONE TERRACE DR,<NA>,SPRING,TX,77379-5333,2113 SAINT CHARLES ST,HOUSTON,77003,1,0
1122,0021480000002,2116 HUTCHINS ST,<NA>,HOUSTON,TX,77003-5934,2116 HUTCHINS ST,HOUSTON,77003,0,0
1189,0021750000003,2202 CAROLINE ST,<NA>,HOUSTON,TX,77002-8906,2202 CAROLINE ST,HOUSTON,77002,0,0
1195,0021750000018,2204 CAROLINE ST,<NA>,HOUSTON,TX,77002-8906,2204 CAROLINE ST,HOUSTON,77002,0,0
1196,0021750000019,2206 CAROLINE ST,<NA>,HOUSTON,TX,77002-8906,2206 CAROLINE ST,HOUSTON,77002,0,0
1233,0021960000001,1418 LIVE OAK ST,<NA>,HOUSTON,TX,77003-4411,1418 LIVE OAK ST,HOUSTON,77003,0,0
1240,0021980000011,2510 LEELAND ST,<NA>,HOUSTON,TX,77003-5204,2510 LEELAND ST,HOUSTON,77003,0,0
1257,0022070000006,1930 CREEGAN PARK COURT,<NA>,HOUSTON,TX,77047-7520,1319 LIVE OAK ST,HOUSTON,77003,0,0
1258,0022070000007,1317 LIVE OAK ST,<NA>,HOUSTON,TX,77003-4408,1317 LIVE OAK ST,HOUSTON,77003,0,0


In [30]:
def normalize_address_text(s):
    return (
        s.astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"[^\w\s#-]", "", regex=True)
    )

sf_candidates["mail_street_norm"] = normalize_address_text(
    sf_candidates["mail_addr_1"]
)

sf_candidates["site_street_norm"] = normalize_address_text(
    sf_candidates["site_addr_1"]
)

In [31]:
sf_candidates["has_both_addresses"] = (
    sf_candidates["mail_street_norm"].notna()
    &
    sf_candidates["site_street_norm"].notna()
    &
    sf_candidates["mail_street_norm"].ne("")
    &
    sf_candidates["site_street_norm"].ne("")
)

sf_candidates["mail_site_same"] = (
    sf_candidates["has_both_addresses"]
    &
    (
        sf_candidates["mail_street_norm"]
        ==
        sf_candidates["site_street_norm"]
    )
)

sf_candidates["mail_site_different"] = (
    sf_candidates["has_both_addresses"]
    &
    ~sf_candidates["mail_site_same"]
)

In [32]:
sf_no_homestead = (
    sf_candidates[
        sf_candidates["any_homestead_evidence"] == 0
    ]
    .copy()
)

print("A1 residential, no homestead:", len(sf_no_homestead))

print(
    "Both mailing + site address available:",
    sf_no_homestead["has_both_addresses"].sum()
)

print(
    "Mailing address = site address:",
    sf_no_homestead["mail_site_same"].sum()
)

print(
    "Mailing address != site address:",
    sf_no_homestead["mail_site_different"].sum()
)

print(
    "Corporate owner:",
    sf_no_homestead["has_corporate_owner"].eq(1).sum()
)

A1 residential, no homestead: 292786
Both mailing + site address available: 292784
Mailing address = site address: 108000
Mailing address != site address: 184784
Corporate owner: 44403


In [33]:
print(
    pd.crosstab(
        sf_no_homestead["mail_site_different"],
        sf_no_homestead["has_corporate_owner"],
        margins=True
    )
)

has_corporate_owner       0      1     All
mail_site_different                       
False                106692   1310  108002
True                 141691  43093  184784
All                  248383  44403  292786


In [34]:
analysis_2021["mail_street_norm"] = normalize_address_text(
    analysis_2021["mail_addr_1"]
)

analysis_2021["site_street_norm"] = normalize_address_text(
    analysis_2021["site_addr_1"]
)

analysis_2021["has_both_addresses"] = (
    analysis_2021["mail_street_norm"].notna()
    &
    analysis_2021["site_street_norm"].notna()
    &
    analysis_2021["mail_street_norm"].ne("")
    &
    analysis_2021["site_street_norm"].ne("")
)

analysis_2021["mail_site_different"] = (
    analysis_2021["has_both_addresses"]
    &
    (
        analysis_2021["mail_street_norm"]
        !=
        analysis_2021["site_street_norm"]
    )
)

In [35]:
analysis_2021["probable_sf_rental"] = (
    analysis_2021["state_class"].eq("A1")
    &
    analysis_2021["in_building_res"].eq(1)
    &
    analysis_2021["any_homestead_evidence"].eq(0)
    &
    analysis_2021["mail_site_different"]
)

analysis_2021["broader_rental"] = (
    analysis_2021["explicit_rental"].eq(1)
    |
    analysis_2021["probable_sf_rental"]
)

In [36]:
print(
    "Explicit rental:",
    analysis_2021["explicit_rental"].sum()
)

print(
    "Probable SF rental:",
    analysis_2021["probable_sf_rental"].sum()
)

print(
    "Overlap:",
    (
        analysis_2021["explicit_rental"].eq(1)
        &
        analysis_2021["probable_sf_rental"]
    ).sum()
)

print(
    "Broader rental universe:",
    analysis_2021["broader_rental"].sum()
)

filing_props = analysis_2021[
    analysis_2021["any_filing_2021"] == 1
]

print(
    "\nFiling properties captured:",
    filing_props["broader_rental"].sum()
)

print(
    "Percent captured:",
    round(
        filing_props["broader_rental"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Matched filings captured:",
    filing_props.loc[
        filing_props["broader_rental"],
        "filing_count_2021"
    ].sum()
)

print(
    "Percent of matched filings captured:",
    round(
        filing_props.loc[
            filing_props["broader_rental"],
            "filing_count_2021"
        ].sum()
        / filing_props["filing_count_2021"].sum()
        * 100,
        2
    ),
    "%"
)

Explicit rental: 16297
Probable SF rental: 184784
Overlap: 643
Broader rental universe: 200438

Filing properties captured: 6146
Percent captured: 72.42 %
Matched filings captured: 23400
Percent of matched filings captured: 87.53 %


In [37]:
model_2021 = (
    analysis_2021[
        analysis_2021["broader_rental"]
    ]
    .copy()
)

print("Modeling properties:", len(model_2021))
print(
    "Properties with filing:",
    model_2021["any_filing_2021"].sum()
)

print(
    "Properties with no linked filing:",
    (model_2021["any_filing_2021"] == 0).sum()
)

print(
    "Total filings represented:",
    model_2021["filing_count_2021"].sum()
)

print(
    "Positive share:",
    round(
        model_2021["any_filing_2021"].mean() * 100,
        2
    ),
    "%"
)

Modeling properties: 200438
Properties with filing: 6146
Properties with no linked filing: 194292
Total filings represented: 23400
Positive share: 3.07 %


In [38]:
model_2021.to_parquet("/content/drive/MyDrive/GNN/model_2021_rental_cohort.parquet",
    index=False
)

In [39]:
owner_edges_2021 = pd.read_parquet(
    "/content/drive/MyDrive/GNN/owner_property_edges_2021_v1.parquet"
)

In [40]:
owner_edges_2021["acct"] = (
    owner_edges_2021["acct"]
    .astype("string")
    .str.zfill(13)
)

model_2021["acct"] = (
    model_2021["acct"]
    .astype("string")
    .str.zfill(13)
)

rental_edges_2021 = owner_edges_2021[
    owner_edges_2021["acct"].isin(model_2021["acct"])
].copy()

print("Rental cohort:", len(model_2021))
print("Properties with owner edge:", rental_edges_2021["acct"].nunique())
print(
    "Owner-edge coverage:",
    round(rental_edges_2021["acct"].nunique() / len(model_2021) * 100, 2),
    "%"
)
print("Unique owners:", rental_edges_2021["owner_id_temp"].nunique())
print("Duplicate property edges:", rental_edges_2021["acct"].duplicated().sum())

Rental cohort: 200438
Properties with owner edge: 168904
Owner-edge coverage: 84.27 %
Unique owners: 115831
Duplicate property edges: 0


In [41]:
#what does the owner edge coverage mean?

In [42]:
edge_accounts = set(rental_edges_2021["acct"])

model_2021["has_owner_edge"] = (
    model_2021["acct"].isin(edge_accounts)
)

coverage_by_label = (
    model_2021
    .groupby("any_filing_2021")["has_owner_edge"]
    .agg(["count", "sum", "mean"])
)

coverage_by_label["mean"] *= 100

print(coverage_by_label)

print(
    "\nPositive properties with owner edge:",
    model_2021.loc[
        model_2021["any_filing_2021"] == 1,
        "has_owner_edge"
    ].sum(),
    "/",
    model_2021["any_filing_2021"].sum()
)

print(
    "Filings on properties with owner edge:",
    model_2021.loc[
        model_2021["has_owner_edge"],
        "filing_count_2021"
    ].sum(),
    "/",
    model_2021["filing_count_2021"].sum()
)

                  count     sum       mean
any_filing_2021                           
0                194292  163307  84.052354
1                  6146    5597  91.067361

Positive properties with owner edge: 5597 / 6146
Filings on properties with owner edge: 22068 / 23400


In [43]:
# --------------------------------------------------
# 1. PROPERTY NODES — all HCAD properties
# --------------------------------------------------

property_nodes_2021 = (
    analysis_2021[["acct"]]
    .drop_duplicates()
    .copy()
)

property_nodes_2021["acct"] = (
    property_nodes_2021["acct"]
    .astype("string")
    .str.zfill(13)
)

property_nodes_2021 = property_nodes_2021.reset_index(drop=True)

property_nodes_2021["property_node_id"] = (
    range(len(property_nodes_2021))
)


# --------------------------------------------------
# 2. OWNER NODES — owners appearing in V1 edges
# --------------------------------------------------

owner_nodes_2021 = (
    owner_edges_2021[["owner_id_temp"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

owner_nodes_2021["owner_node_id"] = (
    range(len(owner_nodes_2021))
)


print("Property nodes:", len(property_nodes_2021))
print("Owner nodes:", len(owner_nodes_2021))

display(property_nodes_2021.head())
display(owner_nodes_2021.head())

Property nodes: 1507914
Owner nodes: 945097


,acct,property_node_id
0,0010010000013,0
1,0010020000001,1
2,0010020000003,2
3,0010020000004,3
4,0010020000013,4


,owner_id_temp,owner_node_id
0,CITY OF HOUSTON,0
1,MILBY CHARLES FAMILY PTNSH,1
2,MICHAEL RYAN FEAGIN TRUST,2
3,BUFFALO BAYOU PARTNERSHIP,3
4,901 COMMERCE LLC,4


In [44]:
edges_2021 = (
    owner_edges_2021
    .merge(
        owner_nodes_2021,
        on="owner_id_temp",
        how="left"
    )
    .merge(
        property_nodes_2021,
        on="acct",
        how="left"
    )
)

print("Owner-property edges:", len(edges_2021))

print(
    "Edges missing owner ID:",
    edges_2021["owner_node_id"].isna().sum()
)

print(
    "Edges missing property ID:",
    edges_2021["property_node_id"].isna().sum()
)

display(
    edges_2021[
        [
            "owner_id_temp",
            "owner_node_id",
            "acct",
            "property_node_id",
            "portfolio_size",
            "has_corporate_owner"
        ]
    ].head()
)


Owner-property edges: 1252988
Edges missing owner ID: 0
Edges missing property ID: 0


,owner_id_temp,owner_node_id,acct,property_node_id,portfolio_size,has_corporate_owner
0,CITY OF HOUSTON,0,0010010000013,0,10382,0
1,MILBY CHARLES FAMILY PTNSH,1,0010020000003,2,5,1
2,MICHAEL RYAN FEAGIN TRUST,2,0010020000004,3,1,0
3,BUFFALO BAYOU PARTNERSHIP,3,0010020000013,4,26,1
4,BUFFALO BAYOU PARTNERSHIP,3,0010020000015,5,26,1


In [45]:
property_feature_cols = [
    "exterior_wall_type_count",
    "has_central_hvac",
    "mean_condition_score",
    "mean_grade_adj",
    "has_slab_foundation",
    "full_baths",
    "mean_cdu_adj",
    "has_pier_foundation",
    "bedrooms",
    "foundation_type_count",
    "total_rooms",
    "fireplace_count",
    "has_cracked_slab",
    "has_pool_spa",
    "has_foundation_repair",
    "has_solar",
    "has_pool",
    "has_porch",
    "has_shed",
    "has_carport",
    "has_outdoor_kitchen",
    "has_garage",
    "half_baths"
]

# Check that all 25 are available
missing_features = [
    col for col in property_feature_cols
    if col not in analysis_2021.columns
]

print("Missing feature columns:", missing_features)

# Check missingness
feature_missingness = (
    analysis_2021[property_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

print(feature_missingness)

Missing feature columns: []
total_rooms                 20.378218
full_baths                  20.372316
bedrooms                    20.369000
mean_cdu_adj                20.366811
has_pier_foundation         20.366082
foundation_type_count       20.366082
has_slab_foundation         20.366082
mean_grade_adj              20.348442
fireplace_count             19.697940
half_baths                  19.697940
mean_condition_score        16.047467
has_central_hvac            15.928627
exterior_wall_type_count    15.926174
has_pool_spa                 0.000000
has_cracked_slab             0.000000
has_foundation_repair        0.000000
has_solar                    0.000000
has_porch                    0.000000
has_pool                     0.000000
has_shed                     0.000000
has_carport                  0.000000
has_outdoor_kitchen          0.000000
has_garage                   0.000000
dtype: float64


In [46]:
draft_feature_cols = [
    col for col in property_feature_cols
    if col not in ["parking_spaces", "max_stories"]
]

rental_missingness = (
    analysis_2021.loc[
        analysis_2021["broader_rental"],
        draft_feature_cols
    ]
    .isna()
    .mean()
    .sort_values(ascending=False)
    * 100
)

comparison = pd.DataFrame({
    "all_HCAD_pct_missing":
        analysis_2021[draft_feature_cols].isna().mean() * 100,

    "rental_cohort_pct_missing":
        analysis_2021.loc[
            analysis_2021["broader_rental"],
            draft_feature_cols
        ].isna().mean() * 100
})

display(
    comparison.sort_values(
        "rental_cohort_pct_missing",
        ascending=False
    )
)

,all_HCAD_pct_missing,rental_cohort_pct_missing
full_baths,20.372316,3.157585
total_rooms,20.378218,3.157086
bedrooms,20.369000,3.148106
foundation_type_count,20.366082,3.142119
has_pier_foundation,20.366082,3.142119
has_slab_foundation,20.366082,3.142119
mean_cdu_adj,20.366811,3.142119
mean_grade_adj,20.348442,3.071274
mean_condition_score,16.047467,0.423572
fireplace_count,19.697940,0.191082


In [47]:
#Property node features
owner_check = (
    owner_edges_2021
    .groupby("owner_id_temp")
    .agg(
        portfolio_values=("portfolio_size", "nunique"),
        corporate_values=("has_corporate_owner", "nunique"),
        portfolio_size=("portfolio_size", "first"),
        has_corporate_owner=("has_corporate_owner", "first")
    )
    .reset_index()
)

print(
    "Owners with inconsistent portfolio_size:",
    (owner_check["portfolio_values"] > 1).sum()
)

print(
    "Owners with inconsistent corporate flag:",
    (owner_check["corporate_values"] > 1).sum()
)

print("Unique owners:", len(owner_check))

Owners with inconsistent portfolio_size: 0
Owners with inconsistent corporate flag: 0
Unique owners: 945097


In [48]:
import numpy as np

owner_features_2021 = owner_check[
    [
        "owner_id_temp",
        "portfolio_size",
        "has_corporate_owner"
    ]
].copy()

owner_features_2021["log_portfolio_size"] = np.log1p(
    owner_features_2021["portfolio_size"]
)

display(owner_features_2021.head())

print(owner_features_2021.shape)
print(owner_features_2021.isna().sum())

,owner_id_temp,portfolio_size,has_corporate_owner,log_portfolio_size
0,0 BELLEAU WOODS TRUST,1,0,0.693147
1,0 CEBRA DEVELOPMENT LLC,3,1,1.386294
2,0 DALLAS ST LLC,1,1,0.693147
3,0 KICKAPOO ROAD LLC,3,1,1.386294
4,0 MADISON STREET DEVELOPMENT LLC,11,1,2.484907


(945097, 4)
owner_id_temp          0
portfolio_size         0
has_corporate_owner    0
log_portfolio_size     0
dtype: int64


In [49]:
# -----------------------------
# PROPERTY NODE TABLE
# -----------------------------

property_node_table = (
    property_nodes_2021
    .merge(
        analysis_2021[
            ["acct"]
            + draft_feature_cols
            + [
                "any_filing_2021",
                "filing_count_2021",
                "broader_rental"
            ]
        ],
        on="acct",
        how="left",
        validate="one_to_one"
    )
    .sort_values("property_node_id")
    .reset_index(drop=True)
)

print("Property node table shape:", property_node_table.shape)
print("Missing labels:", property_node_table["any_filing_2021"].isna().sum())
print("Rental target nodes:", property_node_table["broader_rental"].sum())

print(
    "Positive rental target nodes:",
    (
        property_node_table["broader_rental"]
        &
        property_node_table["any_filing_2021"].eq(1)
    ).sum()
)

Property node table shape: (1507914, 28)
Missing labels: 0
Rental target nodes: 200438
Positive rental target nodes: 6146


In [50]:
# -----------------------------
# OWNER NODE TABLE
# -----------------------------

owner_node_table = (
    owner_nodes_2021
    .merge(
        owner_features_2021,
        on="owner_id_temp",
        how="left",
        validate="one_to_one"
    )
    .sort_values("owner_node_id")
    .reset_index(drop=True)
)

print("Owner node table shape:", owner_node_table.shape)

print(
    "Missing log portfolio size:",
    owner_node_table["log_portfolio_size"].isna().sum()
)

print(
    "Missing corporate flag:",
    owner_node_table["has_corporate_owner"].isna().sum()
)

Owner node table shape: (945097, 5)
Missing log portfolio size: 0
Missing corporate flag: 0


In [51]:
import numpy as np
import pandas as pd

# --------------------------------------------------
# PROPERTY FEATURES
# --------------------------------------------------

X_property = (
    property_node_table[draft_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .copy()
)

# Features that actually have missing values
missing_indicator_cols = [
    col for col in draft_feature_cols
    if X_property[col].isna().any()
]

# Add missingness indicators
for col in missing_indicator_cols:
    X_property[f"{col}_missing"] = (
        X_property[col].isna().astype(np.float32)
    )

# Median imputation for draft graph
for col in draft_feature_cols:
    X_property[col] = X_property[col].fillna(
        X_property[col].median()
    )

X_property = X_property.astype(np.float32)

print("Property feature matrix:", X_property.shape)
print("Remaining NaNs:", X_property.isna().sum().sum())
print("Original features:", len(draft_feature_cols))
print("Missingness indicators:", len(missing_indicator_cols))


# --------------------------------------------------
# OWNER FEATURES
# --------------------------------------------------

X_owner = owner_node_table[
    [
        "log_portfolio_size",
        "has_corporate_owner"
    ]
].astype(np.float32)

print("\nOwner feature matrix:", X_owner.shape)
print("Remaining owner NaNs:", X_owner.isna().sum().sum())

Property feature matrix: (1507914, 36)
Remaining NaNs: 0
Original features: 23
Missingness indicators: 13

Owner feature matrix: (945097, 2)
Remaining owner NaNs: 0


In [52]:
import torch

try:
    from torch_geometric.data import HeteroData
except ImportError:
    !pip -q install torch-geometric
    from torch_geometric.data import HeteroData

In [53]:
# ============================================
# CREATE HETEROGENEOUS GRAPH
# ============================================

data = HeteroData()

# -----------------------------
# PROPERTY NODES
# -----------------------------

data["property"].x = torch.tensor(
    X_property.to_numpy(),
    dtype=torch.float32
)

data["property"].y = torch.tensor(
    property_node_table["any_filing_2021"]
    .astype(int)
    .to_numpy(),
    dtype=torch.long
)

# Primary modeling population:
# explicit rentals + probable SF rentals
data["property"].rental_mask = torch.tensor(
    property_node_table["broader_rental"]
    .astype(bool)
    .to_numpy(),
    dtype=torch.bool
)


# -----------------------------
# OWNER NODES
# -----------------------------

data["owner"].x = torch.tensor(
    X_owner.to_numpy(),
    dtype=torch.float32
)


# -----------------------------
# OWNER -> PROPERTY EDGES
# -----------------------------

owner_ids = torch.tensor(
    edges_2021["owner_node_id"].to_numpy(),
    dtype=torch.long
)

property_ids = torch.tensor(
    edges_2021["property_node_id"].to_numpy(),
    dtype=torch.long
)

edge_index_owner_property = torch.stack(
    [owner_ids, property_ids],
    dim=0
)

data[
    "owner",
    "owns",
    "property"
].edge_index = edge_index_owner_property


# -----------------------------
# PROPERTY -> OWNER EDGES
# -----------------------------

data[
    "property",
    "owned_by",
    "owner"
].edge_index = edge_index_owner_property.flip(0)

In [54]:
print(data)

print(
    "\nProperty x:",
    data["property"].x.shape
)

print(
    "Owner x:",
    data["owner"].x.shape
)

print(
    "Owner -> property edges:",
    data["owner", "owns", "property"].edge_index.shape
)

print(
    "Property -> owner edges:",
    data["property", "owned_by", "owner"].edge_index.shape
)

print(
    "\nRental target nodes:",
    data["property"].rental_mask.sum().item()
)

print(
    "Positive rental nodes:",
    data["property"].y[
        data["property"].rental_mask
    ].sum().item()
)

print(
    "Total positive property nodes:",
    data["property"].y.sum().item()
)

HeteroData(
  property={
    x=[1507914, 36],
    y=[1507914],
    rental_mask=[1507914],
  },
  owner={ x=[945097, 2] },
  (owner, owns, property)={ edge_index=[2, 1252988] },
  (property, owned_by, owner)={ edge_index=[2, 1252988] }
)

Property x: torch.Size([1507914, 36])
Owner x: torch.Size([945097, 2])
Owner -> property edges: torch.Size([2, 1252988])
Property -> owner edges: torch.Size([2, 1252988])

Rental target nodes: 200438
Positive rental nodes: 6146
Total positive property nodes: 8487


In [55]:
from sklearn.model_selection import train_test_split
import numpy as np
import torch

# --------------------------------------------
# TARGET PROPERTY INDICES
# only broader-rental properties
# --------------------------------------------

target_idx = np.flatnonzero(
    property_node_table["broader_rental"]
    .astype(bool)
    .to_numpy()
)

target_y = (
    property_node_table
    .iloc[target_idx]["any_filing_2021"]
    .astype(int)
    .to_numpy()
)

# 70% train / 30% temporary holdout
train_idx, temp_idx = train_test_split(
    target_idx,
    test_size=0.30,
    random_state=42,
    stratify=target_y
)

# Split remaining 30% equally into validation/test
temp_y = (
    property_node_table
    .iloc[temp_idx]["any_filing_2021"]
    .astype(int)
    .to_numpy()
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=temp_y
)

# --------------------------------------------
# BOOLEAN MASKS FOR ALL PROPERTY NODES
# --------------------------------------------

n_properties = len(property_node_table)

train_mask = torch.zeros(n_properties, dtype=torch.bool)
val_mask   = torch.zeros(n_properties, dtype=torch.bool)
test_mask  = torch.zeros(n_properties, dtype=torch.bool)

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data["property"].train_mask = train_mask
data["property"].val_mask = val_mask
data["property"].test_mask = test_mask

# --------------------------------------------
# CHECKS
# --------------------------------------------

for name, mask in [
    ("Train", train_mask),
    ("Validation", val_mask),
    ("Test", test_mask)
]:
    n = mask.sum().item()
    positives = data["property"].y[mask].sum().item()

    print(
        f"{name}: {n:,} nodes | "
        f"{positives:,} positives | "
        f"{positives/n*100:.2f}% positive"
    )

print(
    "\nTotal split nodes:",
    (train_mask | val_mask | test_mask).sum().item()
)

print(
    "Overlap between masks:",
    (
        (train_mask & val_mask).sum()
        + (train_mask & test_mask).sum()
        + (val_mask & test_mask).sum()
    ).item()
)

print(
    "All split nodes are rental targets:",
    bool(
        torch.all(
            data["property"].rental_mask[
                train_mask | val_mask | test_mask
            ]
        )
    )
)

Train: 140,306 nodes | 4,302 positives | 3.07% positive
Validation: 30,066 nodes | 922 positives | 3.07% positive
Test: 30,066 nodes | 922 positives | 3.07% positive

Total split nodes: 200438
Overlap between masks: 0
All split nodes are rental targets: True


In [56]:
# --------------------------------------------
# SCALE PROPERTY FEATURES USING TRAINING NODES
# --------------------------------------------

x_prop = data["property"].x

prop_mean = x_prop[train_mask].mean(dim=0)
prop_std = x_prop[train_mask].std(dim=0)

# Avoid division by zero
prop_std[prop_std == 0] = 1.0

data["property"].x = (
    (x_prop - prop_mean) / prop_std
)

print(
    "Property features finite:",
    torch.isfinite(data["property"].x).all().item()
)

print(
    "Train feature mean (approx):",
    data["property"].x[train_mask].mean().item()
)

print(
    "Train feature std (approx):",
    data["property"].x[train_mask].std().item()
)

Property features finite: True
Train feature mean (approx): -1.556989737139247e-08
Train feature std (approx): 0.999996542930603


In [57]:
# --------------------------------------------
# SCALE OWNER LOG PORTFOLIO SIZE
# --------------------------------------------

owner_x = data["owner"].x.clone()

log_portfolio = owner_x[:, 0]

owner_mean = log_portfolio.mean()
owner_std = log_portfolio.std()

owner_x[:, 0] = (
    (log_portfolio - owner_mean) / owner_std
)

data["owner"].x = owner_x

print(
    "Owner features finite:",
    torch.isfinite(data["owner"].x).all().item()
)

print(
    "Scaled log portfolio mean:",
    data["owner"].x[:, 0].mean().item()
)

print(
    "Scaled log portfolio std:",
    data["owner"].x[:, 0].std().item()
)

Owner features finite: True
Scaled log portfolio mean: -5.914313305765972e-07
Scaled log portfolio std: 0.9999999403953552


In [58]:
import torch.nn.functional as F

from torch.nn import Linear
from torch_geometric.nn import HeteroConv, SAGEConv


class EvictionHeteroGNN(torch.nn.Module):

    def __init__(self, hidden_channels=64):
        super().__init__()

        self.conv1 = HeteroConv(
            {
                ("owner", "owns", "property"):
                    SAGEConv((-1, -1), hidden_channels),

                ("property", "owned_by", "owner"):
                    SAGEConv((-1, -1), hidden_channels),
            },
            aggr="sum"
        )

        self.conv2 = HeteroConv(
            {
                ("owner", "owns", "property"):
                    SAGEConv(
                        (hidden_channels, hidden_channels),
                        hidden_channels
                    ),

                ("property", "owned_by", "owner"):
                    SAGEConv(
                        (hidden_channels, hidden_channels),
                        hidden_channels
                    ),
            },
            aggr="sum"
        )

        self.classifier = Linear(
            hidden_channels,
            1
        )

    def forward(self, x_dict, edge_index_dict):

        x_dict = self.conv1(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            key: F.relu(x)
            for key, x in x_dict.items()
        }

        x_dict = self.conv2(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            key: F.relu(x)
            for key, x in x_dict.items()
        }

        logits = self.classifier(
            x_dict["property"]
        ).squeeze(-1)

        return logits

In [59]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = EvictionHeteroGNN(
    hidden_channels=64
)

print(model)
print("Device:", device)

EvictionHeteroGNN(
  (conv1): HeteroConv(num_relations=2)
  (conv2): HeteroConv(num_relations=2)
  (classifier): Linear(in_features=64, out_features=1, bias=True)
)
Device: cuda


In [60]:
# Save the key graph inputs
property_node_table.to_parquet(
    "/content/drive/MyDrive/GNN/property_node_table_2021.parquet",
    index=False
)

owner_node_table.to_parquet(
    "/content/drive/MyDrive/GNN/owner_node_table_2021.parquet",
    index=False
)

edges_2021[
    [
        "owner_node_id",
        "property_node_id"
    ]
].to_parquet(
    "/content/drive/MyDrive/GNN/gnn_edges_2021.parquet",
    index=False
)

print("Saved graph checkpoints.")

Saved graph checkpoints.


In [61]:
np.save(
    "/content/drive/MyDrive/GNN/train_idx_2021.npy",
    train_idx
)

np.save(
    "/content/drive/MyDrive/GNN/val_idx_2021.npy",
    val_idx
)

np.save(
    "/content/drive/MyDrive/GNN/test_idx_2021.npy",
    test_idx
)

print("Saved split indices.")

Saved split indices.


In [62]:
import torch

print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [63]:
from google.colab import drive
drive.mount("/content/drive")

import numpy as np
import pandas as pd
import torch

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.loader import NeighborLoader
except ImportError:
    !pip -q install torch-geometric
    from torch_geometric.data import HeteroData
    from torch_geometric.loader import NeighborLoader

BASE = "/content/drive/MyDrive/GNN"

# Reload checkpoints
property_node_table = pd.read_parquet(
    f"{BASE}/property_node_table_2021.parquet"
)

owner_node_table = pd.read_parquet(
    f"{BASE}/owner_node_table_2021.parquet"
)

edges_2021 = pd.read_parquet(
    f"{BASE}/gnn_edges_2021.parquet"
)

train_idx = np.load(f"{BASE}/train_idx_2021.npy")
val_idx   = np.load(f"{BASE}/val_idx_2021.npy")
test_idx  = np.load(f"{BASE}/test_idx_2021.npy")

print(property_node_table.shape)
print(owner_node_table.shape)
print(edges_2021.shape)
print(len(train_idx), len(val_idx), len(test_idx))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(1507914, 28)
(945097, 5)
(1252988, 2)
140306 30066 30066


In [64]:
draft_feature_cols = [
    "exterior_wall_type_count",
    "has_central_hvac",
    "mean_condition_score",
    "mean_grade_adj",
    "has_slab_foundation",
    "full_baths",
    "mean_cdu_adj",
    "has_pier_foundation",
    "bedrooms",
    "foundation_type_count",
    "total_rooms",
    "fireplace_count",
    "has_cracked_slab",
    "has_pool_spa",
    "has_foundation_repair",
    "has_solar",
    "has_pool",
    "has_porch",
    "has_shed",
    "has_carport",
    "has_outdoor_kitchen",
    "has_garage",
    "half_baths"
]

# -----------------------------
# PROPERTY FEATURES
# -----------------------------

X_property = (
    property_node_table[draft_feature_cols]
    .apply(pd.to_numeric, errors="coerce")
    .copy()
)

# Missingness indicators
missing_indicator_cols = [
    c for c in draft_feature_cols
    if X_property[c].isna().any()
]

for c in missing_indicator_cols:
    X_property[f"{c}_missing"] = (
        X_property[c].isna().astype(np.float32)
    )

# IMPORTANT: medians from TRAINING target properties only
train_medians = X_property.iloc[train_idx][draft_feature_cols].median()

for c in draft_feature_cols:
    X_property[c] = X_property[c].fillna(train_medians[c])

X_property = X_property.astype(np.float32)

# Convert before scaling
x_prop = torch.tensor(
    X_property.to_numpy(),
    dtype=torch.float32
)

# Scale using training nodes only
prop_mean = x_prop[train_idx].mean(dim=0)
prop_std = x_prop[train_idx].std(dim=0)
prop_std[prop_std == 0] = 1.0

x_prop = (x_prop - prop_mean) / prop_std


# -----------------------------
# OWNER FEATURES
# -----------------------------

x_owner = torch.tensor(
    owner_node_table[
        ["log_portfolio_size", "has_corporate_owner"]
    ].to_numpy(dtype=np.float32),
    dtype=torch.float32
)

# Scale log portfolio only
owner_mean = x_owner[:, 0].mean()
owner_std = x_owner[:, 0].std()

x_owner[:, 0] = (
    x_owner[:, 0] - owner_mean
) / owner_std

print("Property x:", x_prop.shape)
print("Owner x:", x_owner.shape)
print("Property finite:", torch.isfinite(x_prop).all().item())
print("Owner finite:", torch.isfinite(x_owner).all().item())

Property x: torch.Size([1507914, 36])
Owner x: torch.Size([945097, 2])
Property finite: True
Owner finite: True


In [65]:
data = HeteroData()

data["property"].x = x_prop

data["property"].y = torch.tensor(
    property_node_table["any_filing_2021"]
    .astype(int)
    .to_numpy(),
    dtype=torch.float32
)

data["property"].rental_mask = torch.tensor(
    property_node_table["broader_rental"]
    .astype(bool)
    .to_numpy()
)

data["owner"].x = x_owner

owner_ids = torch.tensor(
    edges_2021["owner_node_id"].to_numpy(),
    dtype=torch.long
)

property_ids = torch.tensor(
    edges_2021["property_node_id"].to_numpy(),
    dtype=torch.long
)

edge_index = torch.stack(
    [owner_ids, property_ids],
    dim=0
)

data["owner", "owns", "property"].edge_index = edge_index
data["property", "owned_by", "owner"].edge_index = edge_index.flip(0)

print(data)

HeteroData(
  property={
    x=[1507914, 36],
    y=[1507914],
    rental_mask=[1507914],
  },
  owner={ x=[945097, 2] },
  (owner, owns, property)={ edge_index=[2, 1252988] },
  (property, owned_by, owner)={ edge_index=[2, 1252988] }
)


In [66]:
train_loader = NeighborLoader(
    data,
    input_nodes=(
        "property",
        torch.tensor(train_idx, dtype=torch.long)
    ),
    num_neighbors={
        ("owner", "owns", "property"): [5, 10],
        ("property", "owned_by", "owner"): [5, 10],
    },
    batch_size=2048,
    shuffle=True,
    num_workers=0
)

In [67]:
import torch
import re

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

# PyG wheel naming uses versions such as 2.8.0, 2.9.0, etc.
major_minor = re.match(r"(\d+\.\d+)", torch.__version__).group(1)
torch_version = major_minor + ".0"

cuda_tag = "cu" + torch.version.cuda.replace(".", "")

wheel_url = (
    f"https://data.pyg.org/whl/"
    f"torch-{torch_version}+{cuda_tag}.html"
)

print("Installing from:", wheel_url)

!pip install -q pyg_lib -f {wheel_url}

PyTorch: 2.11.0+cu128
CUDA: 12.8
Installing from: https://data.pyg.org/whl/torch-2.11.0+cu128.html


In [68]:
import pyg_lib

print("pyg-lib installed successfully")

pyg-lib installed successfully


In [69]:
import sys
import importlib.util
import torch
import torch_geometric

from torch_geometric import typing

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("PyG:", torch_geometric.__version__)

print("pyg_lib found:", importlib.util.find_spec("pyg_lib") is not None)
print("torch_sparse found:", importlib.util.find_spec("torch_sparse") is not None)

print("PyG sees pyg_lib:", typing.WITH_PYG_LIB)
print("PyG sees torch_sparse:", typing.WITH_TORCH_SPARSE)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA: 12.8
PyG: 2.8.0.post1
pyg_lib found: True
torch_sparse found: True
PyG sees pyg_lib: True
PyG sees torch_sparse: True


In [70]:
import sys
import subprocess
import torch

# Example:
# torch 2.9.1+cu126 -> 2.9.1
torch_version = torch.__version__.split("+")[0]

# Example:
# CUDA 12.6 -> cu126
cuda_version = torch.version.cuda

if cuda_version is None:
    cuda_tag = "cpu"
else:
    cuda_tag = "cu" + cuda_version.replace(".", "")

wheel_url = (
    f"https://data.pyg.org/whl/"
    f"torch-{torch_version}+{cuda_tag}.html"
)

print("PyTorch:", torch_version)
print("CUDA tag:", cuda_tag)
print("Wheel index:", wheel_url)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--no-index",
    "--find-links",
    wheel_url,
    "pyg_lib",
    "torch_sparse"
])

PyTorch: 2.11.0
CUDA tag: cu128
Wheel index: https://data.pyg.org/whl/torch-2.11.0+cu128.html


0

In [71]:
import torch
import pyg_lib
import torch_geometric

from torch_geometric import typing

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("PyG:", torch_geometric.__version__)
print("pyg_lib imported:", True)
print("PyG sees pyg_lib:", typing.WITH_PYG_LIB)

PyTorch: 2.11.0+cu128
CUDA: 12.8
PyG: 2.8.0.post1
pyg_lib imported: True
PyG sees pyg_lib: True


In [74]:
import torch
import torch_geometric
from torch_geometric import typing

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("PyG:", torch_geometric.__version__)

print("WITH_PYG_LIB:", typing.WITH_PYG_LIB)
print("WITH_TORCH_SPARSE:", typing.WITH_TORCH_SPARSE)

PyTorch: 2.11.0+cu128
CUDA: 12.8
PyG: 2.8.0.post1
WITH_PYG_LIB: True
WITH_TORCH_SPARSE: True


In [75]:
from torch_geometric.loader import NeighborLoader

train_loader = NeighborLoader(
    data,
    input_nodes=(
        "property",
        torch.tensor(train_idx, dtype=torch.long)
    ),
    num_neighbors={
        ("owner", "owns", "property"): [5, 10],
        ("property", "owned_by", "owner"): [5, 10],
    },
    batch_size=2048,
    shuffle=True,
    num_workers=0
)

In [76]:
batch = next(iter(train_loader))

print("Seed properties:", batch["property"].batch_size)
print("Sampled properties:", batch["property"].num_nodes)
print("Sampled owners:", batch["owner"].num_nodes)

print(
    "Owner -> property edges:",
    batch["owner", "owns", "property"].edge_index.shape
)

print(
    "Property -> owner edges:",
    batch["property", "owned_by", "owner"].edge_index.shape
)

Seed properties: 2048
Sampled properties: 5629
Sampled owners: 1636
Owner -> property edges: torch.Size([2, 1731])
Property -> owner edges: torch.Size([2, 5102])


In [77]:
from torch_geometric.loader import NeighborLoader

val_loader = NeighborLoader(
    data,
    input_nodes=(
        "property",
        torch.tensor(val_idx, dtype=torch.long)
    ),
    num_neighbors={
        ("owner", "owns", "property"): [5, 10],
        ("property", "owned_by", "owner"): [5, 10],
    },
    batch_size=4096,
    shuffle=False,
    num_workers=0
)

test_loader = NeighborLoader(
    data,
    input_nodes=(
        "property",
        torch.tensor(test_idx, dtype=torch.long)
    ),
    num_neighbors={
        ("owner", "owns", "property"): [5, 10],
        ("property", "owned_by", "owner"): [5, 10],
    },
    batch_size=4096,
    shuffle=False,
    num_workers=0
)

In [78]:
import torch
import torch.nn.functional as F

from torch.nn import Linear
from torch_geometric.nn import HeteroConv, SAGEConv


class EvictionHeteroGNN(torch.nn.Module):

    def __init__(self, hidden_channels=64):
        super().__init__()

        self.conv1 = HeteroConv(
            {
                ("owner", "owns", "property"):
                    SAGEConv((-1, -1), hidden_channels),

                ("property", "owned_by", "owner"):
                    SAGEConv((-1, -1), hidden_channels),
            },
            aggr="sum"
        )

        self.conv2 = HeteroConv(
            {
                ("owner", "owns", "property"):
                    SAGEConv(
                        (hidden_channels, hidden_channels),
                        hidden_channels
                    ),

                ("property", "owned_by", "owner"):
                    SAGEConv(
                        (hidden_channels, hidden_channels),
                        hidden_channels
                    ),
            },
            aggr="sum"
        )

        self.classifier = Linear(hidden_channels, 1)

    def forward(self, x_dict, edge_index_dict):

        x_dict = self.conv1(x_dict, edge_index_dict)

        x_dict = {
            node_type: F.relu(x)
            for node_type, x in x_dict.items()
        }

        x_dict = self.conv2(x_dict, edge_index_dict)

        x_dict = {
            node_type: F.relu(x)
            for node_type, x in x_dict.items()
        }

        return self.classifier(
            x_dict["property"]
        ).squeeze(-1)

In [79]:
device = torch.device("cuda")

model = EvictionHeteroGNN(
    hidden_channels=64
).to(device)

n_positive = data["property"].y[
    torch.tensor(train_idx)
].sum().item()

n_train = len(train_idx)
n_negative = n_train - n_positive

pos_weight = torch.tensor(
    [n_negative / n_positive],
    dtype=torch.float32,
    device=device
)

criterion = torch.nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print("Positive weight:", pos_weight.item())

Positive weight: 31.614133834838867


In [80]:
batch_size = batch["property"].batch_size

In [81]:
def train_one_epoch():

    model.train()

    total_loss = 0
    total_seeds = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        logits = model(
            batch.x_dict,
            batch.edge_index_dict
        )

        seed_count = batch["property"].batch_size

        seed_logits = logits[:seed_count]

        seed_y = (
            batch["property"].y[:seed_count]
            .float()
        )

        loss = criterion(
            seed_logits,
            seed_y
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * seed_count
        total_seeds += seed_count

    return total_loss / total_seeds

In [82]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)

@torch.no_grad()
def evaluate(loader):

    model.eval()

    all_y = []
    all_prob = []

    for batch in loader:

        batch = batch.to(device)

        logits = model(
            batch.x_dict,
            batch.edge_index_dict
        )

        seed_count = batch["property"].batch_size

        logits = logits[:seed_count]

        y = (
            batch["property"].y[:seed_count]
            .float()
        )

        probs = torch.sigmoid(logits)

        all_y.append(
            y.cpu()
        )

        all_prob.append(
            probs.cpu()
        )

    y_true = torch.cat(all_y).numpy()
    y_prob = torch.cat(all_prob).numpy()

    pr_auc = average_precision_score(
        y_true,
        y_prob
    )

    roc_auc = roc_auc_score(
        y_true,
        y_prob
    )

    return pr_auc, roc_auc

In [83]:
best_val_pr = -1

for epoch in range(1, 11):

    train_loss = train_one_epoch()

    val_pr, val_roc = evaluate(
        val_loader
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Loss {train_loss:.4f} | "
        f"Val PR-AUC {val_pr:.4f} | "
        f"Val ROC-AUC {val_roc:.4f}"
    )

    if val_pr > best_val_pr:

        best_val_pr = val_pr

        torch.save(
            model.state_dict(),
            f"{BASE}/best_gnn_2021_draft.pt"
        )

Epoch 01 | Loss 1.1347 | Val PR-AUC 0.2382 | Val ROC-AUC 0.7655
Epoch 02 | Loss 1.0776 | Val PR-AUC 0.2439 | Val ROC-AUC 0.7656
Epoch 03 | Loss 1.0624 | Val PR-AUC 0.2547 | Val ROC-AUC 0.7766
Epoch 04 | Loss 1.0584 | Val PR-AUC 0.2565 | Val ROC-AUC 0.7747
Epoch 05 | Loss 1.0555 | Val PR-AUC 0.2578 | Val ROC-AUC 0.7744
Epoch 06 | Loss 1.0519 | Val PR-AUC 0.2590 | Val ROC-AUC 0.7752
Epoch 07 | Loss 1.0485 | Val PR-AUC 0.2536 | Val ROC-AUC 0.7717
Epoch 08 | Loss 1.0484 | Val PR-AUC 0.2595 | Val ROC-AUC 0.7721
Epoch 09 | Loss 1.0435 | Val PR-AUC 0.2604 | Val ROC-AUC 0.7726
Epoch 10 | Loss 1.0428 | Val PR-AUC 0.2626 | Val ROC-AUC 0.7736


In [84]:
# Reload the best validation model
model.load_state_dict(
    torch.load(
        f"{BASE}/best_gnn_2021_draft.pt",
        map_location=device
    )
)

model.eval()

test_pr, test_roc = evaluate(test_loader)

print(f"Test PR-AUC:  {test_pr:.4f}")
print(f"Test ROC-AUC: {test_roc:.4f}")

print(
    "PR-AUC / prevalence:",
    round(test_pr / (922 / 30066), 2),
    "x"
)

Test PR-AUC:  0.2550
Test ROC-AUC: 0.7861
PR-AUC / prevalence: 8.31 x
